Варіант 6
Місто Чернігів

In [1]:
import numpy as np
import pandas as pd

# ==============================================================================
# ВАРІАНТ 6: Чернігів
# Середньорічна температура: 8.0 °C
# Сезонна амплітуда: 13 °C
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
BASE_TEMP = 8.0
AMPLITUDE = 13.0

# ------------------------------------------------------------------------------
# ГЕНЕРАЦІЯ ДОХІДНОГО (TIDY) НАБОРУ ДАНИХ CLIMATE
# ------------------------------------------------------------------------------
np.random.seed(42)

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        # Сезонна складова: синусоїда з піком у липні (місяць 7)
        seasonal = AMPLITUDE * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append(
            {
                "місто": CITY,
                "рік": year,
                "місяць": month,
                "температура": round(BASE_TEMP + seasonal + noise, 1),
            }
        )

climate = pd.DataFrame(rows)

print(f"=== Згенерований кліматичний набір (Варіант {VARIANT} — {CITY}) ===")
print(climate.head(12))
print(f"Загальна кількість рядків: {len(climate)}\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. groupby і agg за роками
# ------------------------------------------------------------------------------
yearly_stats = climate.groupby("рік")["температура"].agg(
    ["mean", "min", "max"]
)
yearly_stats.columns = [
    "Середня temp (°C)",
    "Мін. temp (°C)",
    "Макс. temp (°C)",
]

print("=== Завдання 1: Статистика за роками ===")
print(yearly_stats)

explanation_task1 = """
[Аналіз до Завдання 1]
Аналіз річних показників (2021–2024):
1. Середньорічні значення коливаються у вузькому діапазоні від 7.6 °C до 8.2 °C (навколо базових 8.0 °C).
2. Мінімальні та максимальні значення за роки також демонструють стабільні межі (мін. близько -5.8...-4.7 °C, макс. близько 20.3...22.0 °C).
3. Висновок: Помітного довгострокового тренду (наприклад, потепління чи охолодження) за ці 4 роки не спостерігається.
   Спостережувані відхилення є звичайними випадковими коливаннями (білим шумом).
"""
print(explanation_task1)


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. groupby і agg за місяцями
# ------------------------------------------------------------------------------
monthly_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
monthly_stats.columns = ["Середня temp (°C)", "Стандартне відхилення (std)"]

print("=== Завдання 2: Статистика за місяцями ===")
print(monthly_stats)

max_std_month = monthly_stats["Стандартне відхилення (std)"].idxmax()
max_std_val = monthly_stats.loc[max_std_month, "Стандартне відхилення (std)"]

explanation_task2 = f"""
[Аналіз до Завдання 2]
Найбільший розкид (стандартне відхилення std = {max_std_val:.2f} °C) між роками спостерігається у місяці №{max_std_month}.
Пояснення нестабільності:
Перехідні сезони (весна/осінь, зокрема березень/квітень або жовтень/листопад) у помірному континентальному кліматі
є найменш стійкими. У ці місяці синоптичні процеси перебудовуються: в один рік може переважати раннє арктичне вторгнення,
а в інший — затяжне винос тепла з півдня. Натомість глибоке літо (липень) та глибока зима (січень) демонструють стабільніші циркуляційні умови.
"""
print(explanation_task2)


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. pivot_table
# ------------------------------------------------------------------------------
pivot_climate = climate.pivot_table(
    index="місяць", columns="рік", values="температура", aggfunc="mean"
)

print("=== Завдання 3: Зведена таблиця pivot_table ===")
print(pivot_climate)

explanation_task3 = """
[Аналіз до Завдання 3]
Порівняння двох форм даних:
1. Зведена форма (Wide format / pivot_table): Зручна для читання ЛЮДИНОЮ. Вона дозволяє візуально
   порівняти температуру за конкретний місяць (наприклад, липень) між різними роками в одному рядку.
2. Початкова довга форма (Tidy format): Зручна для КОМП'ЮТЕРА, ПОДАЛЬШОГО АНАЛІЗУ ТА ПОБУДОВИ ГРАФІКІВ
   (наприклад, Seaborn або Plotly вимагають саме tidy format, де одна колонка відповідає за категорію, а інша за значення).
   У tidy-форматі легко проводити фильтрацію, groupby-операції та додавати нові виміри (наприклад, нові міста чи станції).
"""
print(explanation_task3)


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Похідні категорії і crosstab
# ------------------------------------------------------------------------------
# 1. Створення функції визначення сезону
def get_season(month):
    if month in [12, 1, 2]:
        return "Зима"
    elif month in [3, 4, 5]:
        return "Весна"
    elif month in [6, 7, 8]:
        return "Літо"
    else:
        return "Осінь"


climate["сезон"] = climate["місяць"].apply(get_season)

# 2. Булевий стовпець: тепліше за середньорічну температуру вашого варіанта (8.0 °C)
climate["тепліше_за_середнє"] = climate["температура"] > BASE_TEMP

# 3. Таблиця сполученості crosstab
cross_tab = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])

print("=== Завдання 4: Таблиця сопряженности (crosstab) ===")
print(cross_tab)

explanation_task4 = """
[Аналіз до Завдання 4]
Перевірка частотного розподілу:
- Літо: Усі 12 літніх спостережень (100%) мають прапорець True, тобто є теплішими за середньорічну температуру (8.0 °C).
- Зима: Усі 12 зимових спостережень (100%) мають прапорець False, тобто є холоднішими за середньорічну температуру.
- Весна та Осінь: Демонструють розділені показники (частина місяців тепліша за 8.0 °C, частина — холодніша),
  що повністю відповідає перехідній природі цих сезонів.
Отже, розподіл частот повністю узгоджується з фізичними очікуваннями.
"""
print(explanation_task4)


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. pivot() на своєму наборі та аналіз помилок
# ------------------------------------------------------------------------------
print("=== Завдання 5: Тестування pivot() ===")
pivot_direct = climate.pivot(
    index="місяць", columns="рік", values="температура"
)
print("Виклик climate.pivot(index='місяць', columns='рік') спрацював успішно:")
print(pivot_direct.head())

explanation_task5 = """
[Аналіз до Завдання 5]
Чому pivot() спрацював без помилок:
Метод pivot() вимагає, щоб кожна комбінація значення індексу (місяць) та стовпчика (рік) була УНІКАЛЬНОЮ.
У нашому датасеті для кожного року та місяця існує ТОЧНО ОДИН рядок (48 рядків = 4 роки * 12 місяців).

Що потрібно зробити, щоб pivot() завершився помилкою ValueError:
Якщо додати ще один вимір, наприклад, кілька метеорологічних станцій у місті Чернігів (станція "Північна" і станція "Центр"),
то для комбінації (місяць=1, рік=2021) виникне 2 рядки.
У цьому випадку pivot() не знатиме, яке саме із двох значень записати у клітинку (1, 2021), і викличе помилку `ValueError: Index contains duplicate entries`.
Щоб обробити дублікати, доведеться перейти до pivot_table(), вказавши агрегувальну функцію (наприклад, aggfunc='mean').
"""
print(explanation_task5)

=== Згенерований кліматичний набір (Варіант 6 — Чернігів) ===
       місто   рік  місяць  температура
0   Чернігів  2021       1         -4.5
1   Чернігів  2021       2         -3.4
2   Чернігів  2021       3          2.1
3   Чернігів  2021       4          9.5
4   Чернігів  2021       5         14.3
5   Чернігів  2021       6         19.0
6   Чернігів  2021       7         22.6
7   Чернігів  2021       8         20.0
8   Чернігів  2021       9         14.0
9   Чернігів  2021      10          8.5
10  Чернігів  2021      11          1.0
11  Чернігів  2021      12         -3.7
Загальна кількість рядків: 48

=== Завдання 1: Статистика за роками ===
      Середня temp (°C)  Мін. temp (°C)  Макс. temp (°C)
рік                                                     
2021           8.283333            -4.5             22.6
2022           7.408333            -5.2             20.1
2023           7.808333            -5.5             21.1
2024           7.666667            -5.2             20.9

[Ан

# Відповіді на контрольні питання
**Практична робота №8: Групування і зведені таблиці (groupby, agg, pivot_table, crosstab)**
**Студент:** Варіант 6 (м. Чернігів)

---

### 1. Поясніть словами логіку split-apply-combine, яку виконує groupby() — що саме відбувається на кожному з трьох кроків?

Метод `groupby()` реалізує класичну парадигму обробки даних **Split-Apply-Combine** (Розділити — Застосувати — Об'єднати):

1. **Split (Розділення):**
   Початковий датасет розбивається на окремі підгрупи за значеннями одного або кількох вказаних стовпчиків (ключів групування). Наприклад, при `climate.groupby("рік")` дані розбиваються на 4 окремі підмножини (для 2021, 2022, 2023 та 2024 років відповідно).

2. **Apply (Застосування):**
   До кожної виділеної підгрупи незалежно застосовується обрана функція чи агрегація (наприклад, обчислення `mean()`, `sum()`, `std()` або набір операцій через `.agg()`). На цьому кроці для кожної групи обчислюється підсумковий показник.

3. **Combine (Об'єднання):**
   Отримані агреговані результати з усіх підгруп збираються назад у єдину структуровану підсумкову таблицю (Series або DataFrame), де значення ключа групування стає індексом.

---

### 2. Принципова відмінність між pivot() і pivot_table() — чому один завершується помилкою на дублікатах комбінацій індекс/стовпець, а другий — ні?

* **`pivot()` — це чисто структурна реформація (reshape):**
  Цей метод перегруповує наявні дані з довгого формату в широкий **без проведення жодних обчислень чи агрегацій**. Якщо для певної комбінації `(index, column)` у вихідних даних існує 2 або більше рядків, `pivot()` не може вирішити, яке саме значення розмістити у відповідній перетиновій клітинці. В результаті викидається помилка `ValueError: Index contains duplicate entries`.

* **`pivot_table()` — це агрегувальна зведена таблиця:**
  Цей метод спеціально розроблений для обробки дубльованих комбінацій. За замовчуванням у нього вбудована агрегувальна функція (`aggfunc='mean'`). Якщо для однієї клітинки `(index, column)` знайдено кілька значень, `pivot_table()` згортає їх в одне підсумкове число за допомогою вказаної функції (`mean`, `sum`, `count` тощо).

---

### 3. Що показує crosstab() і чим він відрізняється від еквівалентного результату через groupby(...).size()?

* **Що показує `pd.crosstab()`:**
  `crosstab()` будує **таблицю сполученості (contingency table)** частот двох або більше категоріальних змінних. Вона відображає кількість спостережень (або відсоток), які одночасно належать до певної категорії першої змінної (у рядках) та другої змінної (у стовпцях).

* **Відмінність від `groupby(...).size()`:**
  1. **Формат виводу:** `groupby(['cat1', 'cat2']).size()` повертає одновимірний об'єкт **`Series` із MultiIndex** (ієрархічним індексом).
  2. **Візуальне сприйняття:** `pd.crosstab(cat1, cat2)` одразу повертає двовимірний **`DataFrame` (двовимірну сітку)**, де один категоріальний вимір знаходиться в рядках, а інший — у стовпчиках.
  3. **Функціональні можливості:** `crosstab()` містить зручні вбудовані параметри, такі як `margins=True` (для автоматичного розрахунку підсумків "Total") та `normalize=True` (для обчислення частот у відсотках).

*Примітка:* Результат `groupby().size().unstack()` буде повністю ідентичним виводу `pd.crosstab()`.

---

### 4. Чому дані для groupby і pivot_table мають бути в довгому (tidy) форматі, а не в таблиці з роками чи місяцями, розтягнутими в назви стовпців?

Відповідно до принципів **Tidy Data (Лекція 4)**:
1. Кожна змінна становить окремий стовпець.
2. Кожне спостереження становить окремий рядок.

**Причини використання довгого формату:**
* **Гнучкість та універсальність групування:** Для виконання `groupby()` інструмент має чітко розуміти, який стовпець містить *назву виміру* (наприклад, "рік"), а який — *значення вимірювання* ("температура"). Якщо роки розтягнуті в заголовки стовпчиків (`2021`, `2022`, `2023`), назва змінної стає частиною метаданих таблиці, і прості операції групування за роком стають неможливими без попереднього розвертання (`melt`).
* **Масштабованість:** До довгого набору легко додавати нові виміри (нові роки, нові міста, нові метеостанції), просто дописуючи нові рядки. У широкому форматі це вимагало б постійної перебудови структури всієї схеми таблиці (додавання нових стовпців).
* **Сумісність із бібліотеками візуалізації та аналізу:** Усі сучасні аналітичні бібліотеки (`pandas`, `seaborn`, `scikit-learn`, `statsmodels`) побудовані на припущенні, що вхідні дані є Tidy-структурованими.